## Deep Dyna-Q Tutorial with CartPole

Adapted from [Deep Dyna-Q: Integrating Planning for Task-Completion Dialogue Policy Learning](https://arxiv.org/abs/1801.06176) (Peng et al., 2018, Microsoft Research). The agent learns a world model from real transitions and uses it to generate simulated experience, supplementing real interactions — both contribute to the Q-network's gradients.

The original paper was coupled to a dialogue NLP task. This implementation generalizes the core algorithm to work with any Gymnasium environment.

![Alt text](best_run.png)

In [1]:
import os
from typing import Any

import gymnasium as gym
import structlog
import torch
import torch.nn as nn
from cart_pole_logger_dyna import CartPoleLoggerDyna
from cart_pole_wm_interpreter import CartPoleInterpreter

from samsara_rl.control.dyna.deep_dyna_q import (
    DeepDynaQ,
)
from samsara_rl.functions.neural_networks.fully_connected import FullyConnected
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

%load_ext tensorboard

In [2]:
cart_pole_wm_interpreter = CartPoleInterpreter()

In [3]:
log = structlog.get_logger()

MAX_EPISODES = 2000
BASE_DIR = "logs/deep_dyna_q"

ALPHA = 0.0005
GAMMA = 0.99
MODEL_PLAN_STEPS = [0, 2, 5]
WARMUPS = [300, 500, 800]

## Q-Network Architecture

Input: 4 values (position, velocity, angle, angular velocity). Output: 2 Q-values (push left, push right).

Two hidden layers (32 → 16) capture the nonlinear relationship between angle and angular velocity. A single hidden layer can work but requires more careful tuning — the angle/angular velocity dynamics involve sine/cosine curves that benefit from the extra representational capacity.

In [4]:
class CartPoleQNetwork(nn.Module):
    def __init__(self, input_dim: int, hidden_dim_1: int, hidden_dim_2: int, output_dim: int) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, output_dim)

    def forward(self, x: Any) -> torch.Tensor:
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu2(out)
        return self.fc3(out)  # type: ignore[no-any-return]

In [5]:
def run_experiment(env, model_plan_steps: int, warmup: int) -> None:
    """Train a Deep Dyna-Q agent with fixed hyperparameters, sweeping plan steps and warmup.

    Args:
        env: Gymnasium CartPole environment.
        model_plan_steps: Number of Q-network training steps on simulated data per real transition.
        warmup: Number of episodes before planning begins.
    """
    model_name = "deep_dyna_q"
    run_name = f"{model_name}_plan_steps={model_plan_steps}_warmup={warmup}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)
    cart_pole_wm = FullyConnected(6, 32, 6, dtype=torch.float32)
    q = CartPoleQNetwork(4, 32, 16, 2)

    agent = DeepDynaQ(
        mdp=env,
        world_model=cart_pole_wm,
        world_model_interpreter=cart_pole_wm_interpreter,
        model_plan_steps=model_plan_steps,
        count_planning_updates=True,
        model_train_steps=10,
        warmup=warmup,
        gamma=GAMMA,
        q=q,
        alpha=ALPHA,
        state_loss=torch.nn.MSELoss(),
        target_update_freq=500,
        epsilon_episodes=800,
        epsilon_min=0.01,
        lr_step_size=100,
        lr_gamma=0.3,
        log_dir=run_dir,
        experiment_name=run_name,
        batch_size=128,
        buffer_size=10000,
    )
    agent.register(CartPoleLoggerDyna())
    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [6]:
def main() -> None:
    """Sweep over plan steps and warmup periods with fixed hyperparameters."""
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for plan_steps in MODEL_PLAN_STEPS:
        for warmup in WARMUPS:
            run_experiment(env, plan_steps, warmup)

In [7]:
main()

2026-11-03 18:30:36 [info     ] starting_run                   run='deep_dyna_q_plan_steps=0_warmup=300'


/home/ashish/reinforcement-learning-library/samsara-rl/src/samsara_rl/control/dqn/dqn_agent.py:162: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.lr_scheduler.step()
/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (

2026-11-03 18:30:44 [debug    ] target_network_swap            episode=50
2026-11-03 18:30:47 [debug    ] target_network_swap            episode=76
2026-11-03 18:30:49 [debug    ] target_network_swap            episode=95
2026-11-03 18:30:53 [debug    ] target_network_swap            episode=108
2026-11-03 18:30:56 [debug    ] target_network_swap            episode=124
2026-11-03 18:30:58 [debug    ] target_network_swap            episode=138
2026-11-03 18:31:02 [debug    ] target_network_swap            episode=154
2026-11-03 18:31:04 [debug    ] target_network_swap            episode=172
2026-11-03 18:31:07 [debug    ] target_network_swap            episode=182
2026-11-03 18:31:09 [debug    ] target_network_swap            episode=195
2026-11-03 18:31:13 [debug    ] target_network_swap            episode=204
2026-11-03 18:31:15 [debug    ] target_network_swap            episode=216
2026-11-03 18:31:17 [debug    ] target_network_swap            episode=225
2026-11-03 18:31:19 [debug  

2026-11-03 18:33:29 [debug    ] target_network_swap            episode=467
2026-11-03 18:33:31 [debug    ] target_network_swap            episode=470
2026-11-03 18:33:33 [debug    ] target_network_swap            episode=472
2026-11-03 18:33:35 [debug    ] target_network_swap            episode=475
2026-11-03 18:33:37 [debug    ] target_network_swap            episode=477
2026-11-03 18:33:40 [debug    ] target_network_swap            episode=479
2026-11-03 18:33:42 [debug    ] target_network_swap            episode=480
2026-11-03 18:33:44 [debug    ] target_network_swap            episode=482
2026-11-03 18:33:46 [debug    ] target_network_swap            episode=486
2026-11-03 18:33:48 [debug    ] target_network_swap            episode=487
2026-11-03 18:33:50 [debug    ] target_network_swap            episode=489
2026-11-03 18:33:53 [debug    ] target_network_swap            episode=491
2026-11-03 18:33:55 [debug    ] target_network_swap            episode=493
2026-11-03 18:33:57 [debu

2026-11-03 18:36:00 [debug    ] target_network_swap            episode=590
2026-11-03 18:36:02 [debug    ] target_network_swap            episode=593
2026-11-03 18:36:04 [debug    ] target_network_swap            episode=594
2026-11-03 18:36:07 [debug    ] target_network_swap            episode=596
2026-11-03 18:36:09 [debug    ] target_network_swap            episode=597
2026-11-03 18:36:13 [debug    ] target_network_swap            episode=600
2026-11-03 18:36:15 [debug    ] target_network_swap            episode=602
2026-11-03 18:36:17 [debug    ] target_network_swap            episode=604
2026-11-03 18:36:19 [debug    ] target_network_swap            episode=605
2026-11-03 18:36:21 [debug    ] target_network_swap            episode=607
2026-11-03 18:36:23 [debug    ] target_network_swap            episode=608
2026-11-03 18:36:25 [debug    ] target_network_swap            episode=611
2026-11-03 18:36:27 [debug    ] target_network_swap            episode=612
2026-11-03 18:36:30 [debu

2026-11-03 18:38:32 [debug    ] target_network_swap            episode=714
2026-11-03 18:38:34 [debug    ] target_network_swap            episode=715
2026-11-03 18:38:36 [debug    ] target_network_swap            episode=717
2026-11-03 18:38:38 [debug    ] target_network_swap            episode=719
2026-11-03 18:38:41 [debug    ] target_network_swap            episode=721
2026-11-03 18:38:43 [debug    ] target_network_swap            episode=723
2026-11-03 18:38:46 [debug    ] target_network_swap            episode=725
2026-11-03 18:38:48 [debug    ] target_network_swap            episode=726
2026-11-03 18:38:50 [debug    ] target_network_swap            episode=728
2026-11-03 18:38:52 [debug    ] target_network_swap            episode=730
2026-11-03 18:38:54 [debug    ] target_network_swap            episode=732
2026-11-03 18:38:56 [debug    ] target_network_swap            episode=734
2026-11-03 18:38:59 [debug    ] target_network_swap            episode=735
2026-11-03 18:39:01 [debu

... (6 outputs left out)

2026-11-03 18:56:08 [debug    ] target_network_swap            episode=1551
2026-11-03 18:56:10 [debug    ] target_network_swap            episode=1552
2026-11-03 18:56:12 [debug    ] target_network_swap            episode=1554
2026-11-03 18:56:14 [debug    ] target_network_swap            episode=1556
2026-11-03 18:56:17 [debug    ] target_network_swap            episode=1558
2026-11-03 18:56:19 [debug    ] target_network_swap            episode=1560
2026-11-03 18:56:21 [debug    ] target_network_swap            episode=1561
2026-11-03 18:56:23 [debug    ] target_network_swap            episode=1563
2026-11-03 18:56:25 [debug    ] target_network_swap            episode=1565
2026-11-03 18:56:27 [debug    ] target_network_swap            episode=1566
2026-11-03 18:56:30 [debug    ] target_network_swap            episode=1568
2026-11-03 18:56:32 [debug    ] target_network_swap            episode=1570
2026-11-03 18:56:34 [debug    ] target_network_swap            episode=1572
2026-11-03 1

2026-11-03 18:58:37 [debug    ] target_network_swap            episode=1671
2026-11-03 18:58:39 [debug    ] target_network_swap            episode=1673
2026-11-03 18:58:42 [debug    ] target_network_swap            episode=1675
2026-11-03 18:58:44 [debug    ] target_network_swap            episode=1676
2026-11-03 18:58:46 [debug    ] target_network_swap            episode=1678
2026-11-03 18:58:48 [debug    ] target_network_swap            episode=1680
2026-11-03 18:58:51 [debug    ] target_network_swap            episode=1682
2026-11-03 18:58:53 [debug    ] target_network_swap            episode=1683
2026-11-03 18:58:55 [debug    ] target_network_swap            episode=1685
2026-11-03 18:58:57 [debug    ] target_network_swap            episode=1687
2026-11-03 18:58:59 [debug    ] target_network_swap            episode=1689
2026-11-03 18:59:01 [debug    ] target_network_swap            episode=1691
2026-11-03 18:59:03 [debug    ] target_network_swap            episode=1692
2026-11-03 1

2026-11-03 19:01:08 [debug    ] target_network_swap            episode=1790
2026-11-03 19:01:11 [debug    ] target_network_swap            episode=1792
2026-11-03 19:01:13 [debug    ] target_network_swap            episode=1794
2026-11-03 19:01:16 [debug    ] target_network_swap            episode=1796
2026-11-03 19:01:18 [debug    ] target_network_swap            episode=1797
2026-11-03 19:01:21 [debug    ] target_network_swap            episode=1799
2026-11-03 19:01:26 [debug    ] target_network_swap            episode=1801
2026-11-03 19:01:28 [debug    ] target_network_swap            episode=1802


KeyboardInterrupt: 

## Analysis 

### Policy and Q-Surface after 50 Episodes
![Alt text](initial_policy.png)

After 50 episodes the agent learns the correct convex shape — states closer to upright are worth more, with the origin as the peak. However the relative difference in Q-values between actions is heavily skewed. It can't distinguish when to push left vs right.

### Policy and Q-Surface after 800 Episodes
![Alt text](policy.png)

After 800 episodes the action split converges close to optimal — push left when leaning right, push right when leaning left. Learning rate decay was crucial; without it the max(Q, a') step causes Q-values to spiral due to [maximization bias](https://en.wikipedia.org/wiki/Reinforcement_learning#Maximization_bias).

### No Alpha Decay — Maximization Bias
![Alt text](no_alpha_decay.png)

Without aggressive LR decay, Q-estimates spiral and one action dominates everywhere. The tabletop shape of the Q-surface shows the network treats all non-edge states as equally good — a well-trained network would have gradual curvature. The fix is either decaying the learning rate or tuning gamma to compress the future reward range enough for the agent to distinguish good from bad states.